# P2 Unit Tests — Implementation Verification

This notebook tests every function in `set_ops.py` and `correlation.py` using
**synthetic, deterministic data** so you can verify correctness without needing
Person 1's actual pipeline output.

Each section prints `PASS ✓` or `FAIL ✗` with a short explanation.

Run all cells top-to-bottom and send back the output — if all tests pass we move on.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import tempfile, json

# ── Add src/ to path ──────────────────────────────────────────────────────────
REPO_ROOT = Path("__file__").resolve().parent.parent
SRC_DIR   = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Simple test harness ───────────────────────────────────────────────────────
_results = []

def check(name: str, condition: bool, detail: str = "") -> None:
    icon = "PASS ✓" if condition else "FAIL ✗"
    msg  = f"{icon}  {name}" + (f" — {detail}" if detail else "")
    _results.append((condition, name))
    print(msg)

def section(title: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)

def final_summary() -> None:
    passed = sum(1 for ok, _ in _results if ok)
    total  = len(_results)
    print(f"\n{'='*60}")
    print(f"  TOTAL: {passed}/{total} tests passed")
    if passed == total:
        print("  ALL TESTS PASSED ✓")
    else:
        failed = [name for ok, name in _results if not ok]
        print("  FAILED:", failed)
    print('='*60)

print("Test harness ready. SRC_DIR =", SRC_DIR)

---
## 1 · set_ops.py — core math

In [ ]:
from set_ops import (
    jaccard,
    expected_random_jaccard,
    compute_pairwise_intersections,
    compute_triple_intersection,
    compute_mixed_intersections,
    load_address_set,
    build_full_set_intersection_report,
    build_cross_window_set_summary,
    save_set_report,
)
print("set_ops imported OK")

In [ ]:
section("1a — jaccard()")

# Identical sets → 1.0
s = frozenset(["0xaaa", "0xbbb", "0xccc"])
check("identical sets → 1.0", jaccard(s, s) == 1.0)

# Disjoint sets → 0.0
a = frozenset(["0xaaa", "0xbbb"])
b = frozenset(["0xccc", "0xddd"])
check("disjoint sets → 0.0", jaccard(a, b) == 0.0)

# Overlap of 1 out of 4 unique → 1/4 = 0.25
a = frozenset(["0x001", "0x002"])
b = frozenset(["0x002", "0x003"])
j = jaccard(a, b)
check("50% overlap → 0.333...", abs(j - 1/3) < 1e-9, f"got {j:.6f}")

# Both empty → 0.0
check("both empty → 0.0", jaccard(frozenset(), frozenset()) == 0.0)

In [ ]:
section("1b — expected_random_jaccard()")

# With EVM address space (2^160) two tiny sets should have near-zero expected Jaccard
exp = expected_random_jaccard(1000, 1000)
check("expected Jaccard (1k,1k) < 1e-40", exp < 1e-40, f"got {exp:.3e}")

# With a tiny address space the math should be non-trivial
exp_small = expected_random_jaccard(3, 3, address_space=10)
# E[|A∩B|] ≈ 9/10 = 0.9  →  E[J] ≈ 0.9 / (3+3-0.9) ≈ 0.176
check("expected Jaccard with small space is non-zero", exp_small > 0, f"got {exp_small:.4f}")

# Edge cases
check("size_a=0 → 0.0", expected_random_jaccard(0, 1000) == 0.0)
check("address_space=0 → 0.0", expected_random_jaccard(100, 100, address_space=0) == 0.0)

In [ ]:
section("1c — compute_pairwise_intersections()")

eth  = frozenset(["0x001", "0x002", "0x003"])
poly = frozenset(["0x002", "0x003", "0x004"])
bnb  = frozenset(["0x003", "0x005", "0x006"])

df_pw = compute_pairwise_intersections(
    {"ethereum": eth, "polygon": poly, "bnb": bnb},
    label="active",
)

check("returns 3 rows (C(3,2)=3)", len(df_pw) == 3, f"got {len(df_pw)}")
check("has jaccard column", "jaccard" in df_pw.columns)
check("has intersection_size column", "intersection_size" in df_pw.columns)

# eth ∩ poly = {0x002, 0x003} → |inter|=2, |union|=4 → J=0.5
row_ep = df_pw[
    df_pw["chain_a"].isin(["ethereum","polygon"]) &
    df_pw["chain_b"].isin(["ethereum","polygon"])
].iloc[0]
check("ETH ∩ POLY intersection_size = 2", row_ep["intersection_size"] == 2)
check("ETH ∩ POLY jaccard = 0.5", abs(row_ep["jaccard"] - 0.5) < 1e-9)

# enrichment must be positive when Jaccard > 0
check("enrichment is positive where Jaccard > 0",
      (df_pw["jaccard"] > 0).all() == (df_pw["jaccard_enrichment"] > 0).all())

display(df_pw[["chain_a","chain_b","intersection_size","jaccard","jaccard_enrichment"]])

In [ ]:
section("1d — compute_triple_intersection()")

eth  = frozenset(["0x001", "0x002", "0x003", "0x007"])
poly = frozenset(["0x002", "0x003", "0x004"])
bnb  = frozenset(["0x003", "0x005", "0x006"])

triple = compute_triple_intersection(
    {"ethereum": eth, "polygon": poly, "bnb": bnb}
)

check("triple intersection of 3 chains returns non-empty dict", bool(triple))
# Only 0x003 is in all three
check("triple intersection size = 1 (only 0x003)", triple["intersection_size"] == 1)
check("intersection_addresses contains 0x003", "0x003" in triple["intersection_addresses"])
check("jaccard is a float in [0,1]", 0.0 <= triple["jaccard"] <= 1.0)

# Only 2 chains → should return empty dict
triple2 = compute_triple_intersection({"ethereum": eth, "polygon": poly})
check("<3 chains → empty dict", triple2 == {})

print("Triple result:", {k: v for k, v in triple.items() if k != "intersection_addresses"})

In [ ]:
section("1e — compute_mixed_intersections()")

active_sets  = {
    "ethereum": frozenset(["0x001", "0x002", "0x003"]),
    "polygon":  frozenset(["0x003", "0x004"]),
}
passive_sets = {
    "ethereum": frozenset(["0x010", "0x011"]),
    "polygon":  frozenset(["0x001", "0x005"]),
}

df_mixed = compute_mixed_intersections(active_sets, passive_sets)

# 2 chains → 2*(2-1) = 2 ordered pairs
check("2 chains → 2 rows", len(df_mixed) == 2, f"got {len(df_mixed)}")
check("has active_chain column", "active_chain" in df_mixed.columns)
check("jaccard values in [0,1]", (df_mixed["jaccard"] >= 0).all() and (df_mixed["jaccard"] <= 1).all())

# active[ETH]={0x001,0x002,0x003} ∩ passive[POLY]={0x001,0x005} → {0x001} → inter=1, union=5, J=0.2
row_eth_poly = df_mixed[
    (df_mixed["active_chain"] == "ethereum") &
    (df_mixed["passive_chain"] == "polygon")
].iloc[0]
check("active[ETH] ∩ passive[POLY] = 1", row_eth_poly["intersection_size"] == 1)
check("active[ETH] ∩ passive[POLY] jaccard = 0.2", abs(row_eth_poly["jaccard"] - 0.2) < 1e-9)

display(df_mixed[["active_chain","passive_chain","intersection_size","jaccard"]])

In [ ]:
section("1f — load_address_set()")

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "active.csv"
    df_test = pd.DataFrame({"address": ["0xAAA", "0xBBB", "0xCCC", None]})
    df_test.to_csv(p, index=False)

    loaded = load_address_set(p)
    check("loads 3 valid addresses", len(loaded) == 3, f"got {len(loaded)}")
    check("addresses are lower-cased", "0xaaa" in loaded)
    check("None values are dropped", None not in loaded)

    # Non-existent file → empty frozenset
    missing = load_address_set(Path(tmpdir) / "ghost.csv")
    check("missing file → empty frozenset", len(missing) == 0)

In [ ]:
section("1g — build_full_set_intersection_report()")

# Build a fake interim directory with 3 chains × 1 window
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)
    window = "1blk"
    chains = ["ethereum", "polygon", "bnb"]

    # Addresses deliberately overlapping
    datasets = {
        "ethereum": {"active": ["0x001","0x002","0x003"], "passive": ["0x010","0x011"]},
        "polygon":  {"active": ["0x002","0x003","0x004"], "passive": ["0x001","0x012"]},
        "bnb":      {"active": ["0x003","0x005","0x006"], "passive": ["0x007","0x013"]},
    }
    for chain, data in datasets.items():
        d = root / window / chain
        d.mkdir(parents=True)
        pd.DataFrame({"address": data["active"]}).to_csv(d / "active_eoa_addresses.csv", index=False)
        pd.DataFrame({"address": data["passive"]}).to_csv(d / "passive_eoa_addresses.csv", index=False)

    report = build_full_set_intersection_report(
        interim_dir=root,
        window_label=window,
        chains=chains,
    )

    check("report has 6 keys", len(report) == 6)
    check("active_pairwise is a DataFrame", isinstance(report["active_pairwise"], pd.DataFrame))
    check("active_pairwise has 3 rows (C(3,2))", len(report["active_pairwise"]) == 3)
    check("passive_pairwise has 3 rows", len(report["passive_pairwise"]) == 3)
    check("active_triple is a non-empty dict", bool(report["active_triple"]))
    check("only 0x003 in active triple intersection",
          report["active_triple"]["intersection_size"] == 1)
    check("mixed has 6 rows (2 directed pairs per chain)",
          len(report["mixed"]) == 6)  # 3*2=6 ordered pairs
    check("summary DataFrame is non-empty", not report["summary"].empty)

    # Test save_set_report doesn't crash
    out_dir = root / "outputs"
    save_set_report(report, out_dir, window)
    expected_csvs = [
        out_dir / f"set_intersections_active_{window}.csv",
        out_dir / f"set_intersections_passive_{window}.csv",
        out_dir / f"set_intersections_mixed_{window}.csv",
        out_dir / f"set_intersections_summary_{window}.csv",
    ]
    for csv_path in expected_csvs:
        check(f"file exists: {csv_path.name}", csv_path.exists())

---
## 2 · correlation.py — core functions

In [ ]:
from correlation import (
    temporal_delta_analysis,
    run_all_temporal,
    volume_correlation,
    run_all_volume,
    frequency_correlation_from_features,
    run_all_frequency,
    active_passive_ratio_analysis,
    run_all_correlations,
    build_cross_window_correlation_summary,
    save_correlation_results,
)
print("correlation imported OK")

In [ ]:
# ── Synthetic feature table fixture ──────────────────────────────────────────
# 5 addresses on both ethereum and polygon, 3 only on bnb
rng = np.random.default_rng(42)

def _make_feature_row(address, chain, first_ts, tx_count, seed):
    r = np.random.default_rng(seed)
    return {
        "address": address,
        "chain": chain,
        "first_seen_ts": first_ts,
        "last_seen_ts": first_ts + int(r.integers(1000, 100000)),
        "total_tx_count": tx_count,
        "recent_7d_tx_count": max(0, tx_count - int(r.integers(0, tx_count+1))),
        "recent_30d_tx_count": max(0, tx_count - int(r.integers(0, tx_count//2+1))),
        "recent_90d_tx_count": max(0, tx_count - int(r.integers(0, tx_count//4+1))),
        "active_days": int(r.integers(1, 50)),
        "activity_density": float(r.uniform(0, 1)),
    }

rows = []
base_ts = 1_700_000_000  # 2023-11 epoch timestamp

for i in range(5):  # 5 overlap addresses
    addr = f"0x{i:040x}"
    eth_ts   = base_ts + i * 86400          # ethereum: each 1 day apart
    poly_ts  = base_ts + i * 86400 + 3600   # polygon:  1 hour after eth
    eth_tx   = 10 + i * 5
    poly_tx  = 8  + i * 4  # correlated with eth
    rows.append(_make_feature_row(addr, "ethereum", eth_ts,  eth_tx,  seed=i*2))
    rows.append(_make_feature_row(addr, "polygon",  poly_ts, poly_tx, seed=i*2+1))

for i in range(3):  # 3 bnb-only addresses
    addr = f"0xbnb{i:037x}"
    rows.append(_make_feature_row(addr, "bnb", base_ts + i*7200, 5+i*3, seed=100+i))

DF_FEAT = pd.DataFrame(rows)
print(f"Synthetic feature table: {len(DF_FEAT)} rows, chains: {DF_FEAT['chain'].unique()}")
display(DF_FEAT.head(8))

In [ ]:
section("2a — temporal_delta_analysis()")

result = temporal_delta_analysis(DF_FEAT, "ethereum", "polygon")

check("returns dict", isinstance(result, dict))
check("n_overlap = 5", result["n_overlap"] == 5, f"got {result['n_overlap']}")
check("deltas_days is a numpy array", isinstance(result["deltas_days"], np.ndarray))
check("deltas_days has 5 elements", len(result["deltas_days"]) == 5)

# polygon timestamps are always 1 hour (1/24 day) after ethereum
# sign convention: positive delta → chain_a (ethereum) was first
# delta = ts_b - ts_a = poly_ts - eth_ts = +3600 s = +0.0417 days → positive (eth first)
check("all deltas positive (ETH was first)", (result["deltas_days"] > 0).all(),
      f"min delta={result['deltas_days'].min():.4f}")

check("abs_median_delta ≈ 1/24 day (1h)",
      abs(result["abs_median_delta_days"] - 1/24) < 0.01,
      f"got {result['abs_median_delta_days']:.4f}")

check("pearson_r is returned", "pearson_r" in result)
check("spearman_r is returned", "spearman_r" in result)

# ETH ∩ BNB: no overlap → n_overlap = 0
result_no_overlap = temporal_delta_analysis(DF_FEAT, "ethereum", "bnb")
check("ETH ∩ BNB: n_overlap = 0", result_no_overlap["n_overlap"] == 0)

print(f"  Median Δ ETH→POLY: {result['abs_median_delta_days']:.4f} days")
print(f"  pearson_r: {result['pearson_r']}")

In [ ]:
section("2b — run_all_temporal()")

df_temporal = run_all_temporal(DF_FEAT, chains=["ethereum", "polygon", "bnb"])

check("returns DataFrame", isinstance(df_temporal, pd.DataFrame))
check("3 rows (C(3,2)=3 pairs)", len(df_temporal) == 3, f"got {len(df_temporal)}")
check("has n_overlap column", "n_overlap" in df_temporal.columns)
check("ETH-POLY overlap = 5",
      df_temporal[df_temporal["chain_a"]=="ethereum"]["n_overlap"].iloc[0] == 5)

display(df_temporal)

In [ ]:
section("2c — volume_correlation()")

result = volume_correlation(DF_FEAT, "ethereum", "polygon", volume_col="total_tx_count")

check("returns dict", isinstance(result, dict))
check("n_overlap = 5", result["n_overlap"] == 5)
check("pearson_r is present", result.get("pearson_r") is not None)
check("spearman_r is present", result.get("spearman_r") is not None)
check("pearson_r_log is present", result.get("pearson_r_log") is not None)

# ETH tx=10,15,20,25,30 and POLY tx=8,12,16,20,24 — perfectly linear → r should be ≈1
check("pearson_r ≈ 1.0 (perfectly correlated data)",
      abs(result["pearson_r"] - 1.0) < 1e-6,
      f"got {result['pearson_r']:.6f}")

# No overlap → n_overlap < 3, should return partial result without crashing
result_empty = volume_correlation(DF_FEAT, "ethereum", "bnb")
check("no-overlap case returns dict without crash", isinstance(result_empty, dict))
check("no-overlap: n_overlap = 0", result_empty["n_overlap"] == 0)

print(f"  pearson_r: {result['pearson_r']:.6f}")
print(f"  spearman_r: {result['spearman_r']:.6f}")

In [ ]:
section("2d — run_all_volume()")

df_volume = run_all_volume(DF_FEAT, chains=["ethereum", "polygon", "bnb"])

check("returns DataFrame", isinstance(df_volume, pd.DataFrame))
check("3 rows (C(3,2))", len(df_volume) == 3)
check("volume_col column exists", "volume_col" in df_volume.columns)

display(df_volume)

In [ ]:
section("2e — frequency_correlation_from_features()")

result = frequency_correlation_from_features(DF_FEAT, "ethereum", "polygon")

check("returns dict", isinstance(result, dict))
check("n_overlap = 5", result["n_overlap"] == 5, f"got {result['n_overlap']}")
check("mean_per_address_r is a float",
      isinstance(result.get("mean_per_address_r"), float),
      f"got {type(result.get('mean_per_address_r'))}")
check("mean_per_address_r in [-1, 1]",
      -1.0 <= result["mean_per_address_r"] <= 1.0,
      f"got {result['mean_per_address_r']:.4f}")
check("per_address_r_distribution is numpy array",
      isinstance(result.get("per_address_r_distribution"), np.ndarray))
check("bucket_results is a DataFrame",
      isinstance(result.get("bucket_results"), pd.DataFrame))
check("bucket_results is non-empty", not result["bucket_results"].empty)

# No-overlap case
result_no = frequency_correlation_from_features(DF_FEAT, "ethereum", "bnb")
check("no-overlap: n_overlap = 0", result_no["n_overlap"] == 0)

print(f"  mean_per_address_r: {result['mean_per_address_r']:.4f}")
display(result["bucket_results"])

In [ ]:
section("2f — active_passive_ratio_analysis()")

active_s  = {
    "ethereum": frozenset(["0x001","0x002","0x003","0x004"]),
    "polygon":  frozenset(["0x001","0x002","0x005"]),
    "bnb":      frozenset(["0x006","0x007"]),
}
passive_s = {
    "ethereum": frozenset(["0x010","0x011"]),
    "polygon":  frozenset(["0x003","0x012"]),
    "bnb":      frozenset(["0x001","0x013"]),
}

df_ratio = active_passive_ratio_analysis(active_s, passive_s)

check("returns DataFrame", isinstance(df_ratio, pd.DataFrame))
check("3 chains × 2 targets = 6 rows", len(df_ratio) == 6, f"got {len(df_ratio)}")
check("has share_active_any_role column", "share_active_any_role" in df_ratio.columns)
check("enrichment_factor >= 0", (df_ratio["enrichment_factor"] >= 0).all())

# active[ETH]={0x001,0x002,0x003,0x004}
# all[POLY]={0x001,0x002,0x005,0x003,0x012}
# overlap = {0x001,0x002,0x003} → share_any = 3/4 = 0.75
row_ep = df_ratio[
    (df_ratio["source_chain"] == "ethereum") &
    (df_ratio["target_chain"] == "polygon")
].iloc[0]
check("active[ETH] ∩ any[POLY]: n_any_on_target = 3",
      row_ep["n_any_on_target"] == 3,
      f"got {row_ep['n_any_on_target']}")
check("share_active_any_role = 0.75",
      abs(row_ep["share_active_any_role"] - 0.75) < 1e-6,
      f"got {row_ep['share_active_any_role']:.4f}")

display(df_ratio)

In [ ]:
section("2g — run_all_correlations() integration test")

active_s2  = {
    "ethereum": frozenset(DF_FEAT[DF_FEAT["chain"]=="ethereum"]["address"]),
    "polygon":  frozenset(DF_FEAT[DF_FEAT["chain"]=="polygon"]["address"]),
    "bnb":      frozenset(DF_FEAT[DF_FEAT["chain"]=="bnb"]["address"]),
}
passive_s2 = {c: frozenset() for c in active_s2}  # empty passives for this test

results = run_all_correlations(
    feature_df=DF_FEAT,
    active_sets=active_s2,
    passive_sets=passive_s2,
    chains=["ethereum", "polygon", "bnb"],
)

check("result has 4 keys", set(results.keys()) == {"temporal","volume","frequency","ratio"})
check("temporal is DataFrame", isinstance(results["temporal"], pd.DataFrame))
check("volume is DataFrame", isinstance(results["volume"], pd.DataFrame))
check("frequency is DataFrame", isinstance(results["frequency"], pd.DataFrame))
check("ratio is DataFrame (active_sets provided)", isinstance(results["ratio"], pd.DataFrame))

# Without active_sets → ratio should be None
results_no_ratio = run_all_correlations(DF_FEAT, chains=["ethereum", "polygon"])
check("ratio is None when sets not provided", results_no_ratio["ratio"] is None)

print("Temporal:")
display(results["temporal"])

In [ ]:
section("2h — save_correlation_results()")

with tempfile.TemporaryDirectory() as tmpdir:
    out = Path(tmpdir)
    save_correlation_results(results, out, "1blk")
    expected = [
        out / "corr_temporal_1blk.csv",
        out / "corr_volume_1blk.csv",
        out / "corr_frequency_1blk.csv",
        out / "corr_ratio_1blk.csv",
    ]
    for f in expected:
        check(f"saved: {f.name}", f.exists())

In [ ]:
section("2i — build_cross_window_correlation_summary()")

all_results = {
    "1blk":    results,
    "10blk":   results,  # reuse same data; just checking stacking logic
}

cross = build_cross_window_correlation_summary(all_results)

check("returns dict with 4 keys", set(cross.keys()) == {"temporal","volume","frequency","ratio"})
check("temporal has 'window' column", "window" in cross["temporal"].columns)
check("temporal has 2×3 = 6 rows (2 windows × 3 pairs)",
      len(cross["temporal"]) == 6, f"got {len(cross['temporal'])}")

display(cross["temporal"][["window","chain_a","chain_b","n_overlap","pearson_r"]])

---
## 3 · classify.py + normalize.py (smoke tests)

In [ ]:
section("3 — classify_from_normalized_df smoke test")

from normalize import normalize_feature_table
from classify  import classify_from_normalized_df, behavioral_group_counts

# Build a minimal feature table for normalize → classify
_rows = []
for i in range(10):
    addr  = f"0x{i:040x}"
    chain = "ethereum" if i < 6 else "polygon"
    _rows.append({
        "address": addr,
        "chain": chain,
        "chain_id": 1 if chain == "ethereum" else 137,
        "snapshot_ts": 1700000000,
        "observation_window_blocks": 1000,
        "native_tx_count": i * 3,
        "erc20_transfer_count": i,
        "internal_tx_count": i // 2,
        "total_tx_count": i * 4,
        "first_seen_ts": 1_690_000_000 + i * 86400,
        "last_seen_ts":  1_700_000_000 - i * 86400,
        "lifetime_days": float(i * 10 + 1),
        "days_since_last_active": float(i * 5),
        "active_days": max(1, i * 2),
        "active_weeks": max(1, i),
        "activity_density": min(1.0, i * 0.05),
        "distinct_counterparties": i * 2 + 1,
        "distinct_outgoing_counterparties": i + 1,
        "distinct_incoming_counterparties": i,
        "distinct_contracts": i,
        "outgoing_ratio": 0.5,
        "contract_interaction_ratio": 0.3,
        "erc20_ratio": 0.2,
        "internal_ratio": 0.1,
        "mean_gap_seconds": float(3600 * (i+1)),
        "median_gap_seconds": float(3600 * i + 1),
        "mean_gas_used": 21000.0,
        "median_gas_used": 21000.0,
        "mean_gas_price_wei": 20e9,
        "median_gas_price_wei": 20e9,
        "recent_7d_tx_count": max(0, i - 2),
        "recent_30d_tx_count": max(0, i * 2 - 1),
        "recent_90d_tx_count": max(0, i * 3),
        "has_activity": i > 0,
        "is_query_truncated": False,
    })

df_small = pd.DataFrame(_rows)

try:
    df_classified = classify_from_normalized_df(df_small)
    check("classify_from_normalized_df runs without error", True)
    check("returns a DataFrame", isinstance(df_classified, pd.DataFrame))
    check("has behavioral_group column", "behavioral_group" in df_classified.columns)
    check("no NaN behavioral_group values", df_classified["behavioral_group"].notna().all())

    counts = behavioral_group_counts(df_classified)
    check("behavioral_group_counts returns DataFrame", isinstance(counts, pd.DataFrame))
    check("share column sums to 1", abs(counts["share"].sum() - 1.0) < 1e-9)

    print("\nBehavioral group counts:")
    display(counts)

except Exception as exc:
    check("classify_from_normalized_df runs without error", False, str(exc))

---
## 4 · Final summary

In [ ]:
final_summary()